In [82]:
import os
import sys

# Add the project root to the path
PROJECT_ROOT = '/Users/lekiaprosper/Documents/CoMoChEng/Prometheus/kineticmodelssite'
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Load environment variables from .env.dev
from dotenv import load_dotenv
load_dotenv(os.path.join(PROJECT_ROOT, '.env.dev'))

# Configure Django settings
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'kms.settings')

# Allow Django ORM in async context (Jupyter notebooks)
os.environ['DJANGO_ALLOW_ASYNC_UNSAFE'] = 'true'

import django
django.setup()

# Fix duplicate output caused by colorama (initialized by Django)
try:
    import colorama
    colorama.deinit()
except:
    pass


In [83]:
from chemked_database.models import *
from database.models import *
from database.services.exports import build_cantera_yaml

In [84]:
datasets = ExperimentDataset.objects.all()
ignition_delay_datasets = datasets.filter(experiment_type='ignition delay')
for dataset in ignition_delay_datasets:
    print(dataset)
    # Add your processing logic here

x_Cooke_1971_2009 (13 points)
n-heptane/Vermeer 1972/st_vermeer_1972.yaml (72 points)
x_Bowman_1975_07_x (4 points)
x_Bowman_1975_2001 (6 points)
x_Bowman_1975_2002 (6 points)
x_Bowman_1975_2003 (6 points)
x_Bowman_1975_2004 (6 points)
x_Bowman_1975_2005 (6 points)
x_Bowman_1975_2006 (5 points)
x_Bowman_1975_2008 (4 points)
n-heptane/Burcat 1981/st_burcat_1981-1.yaml (18 points)
n-heptane/Burcat 1981/st_burcat_1981-2.yaml (6 points)
n-heptane/Burcat 1981/st_burcat_1981-3.yaml (6 points)
n-heptane/Burcat 1981/st_burcat_1981-4.yaml (6 points)
n-heptane/Burcat 1981/st_burcat_1981-5.yaml (6 points)
x_Natarajan_1981_2051 (8 points)
x_Natarajan_1981_2052 (8 points)
x_Natarajan_1981_2053 (9 points)
x_Natarajan_1981_2054 (8 points)
x_Natarajan_1981_2055 (6 points)
x_Natarajan_1981_2056 (7 points)
x_Tsuboi_1981_2029 (4 points)
x_Tsuboi_1981_2030 (4 points)
x_Tsuboi_1981_2031 (4 points)
x_Tsuboi_1981_2032 (4 points)
x_Tsuboi_1981_2033 (4 points)
x_Tsuboi_1981_2034 (5 points)
x_Tsuboi_1981_2035 (

In [85]:
# Get a single dataset object (not a QuerySet)
n_heptane = ignition_delay_datasets.filter(chemked_file_path='n-heptane/Burcat 1981/st_burcat_1981-1.yaml').first()

In [86]:
apparatus = n_heptane.apparatus
print(f"Apparatus: {apparatus.kind}")

Apparatus: shock tube


In [87]:
# Access the fuel_species property (not a method)
fuel_species = n_heptane.fuel_species[0]
print(f"Fuel Species: {fuel_species}")

Fuel Species: nC7H16


In [88]:
datapoints = n_heptane.datapoints.all()
species_composition = {}
for point in datapoints:
    # print(point.composition.species.all())
    for species in point.composition.species.all():
        species_composition[species.species_name] = species.amount
        
print(species_composition)

{'Ar': 0.88, 'O2': 0.11, 'nC7H16': 0.01}


In [89]:
import pyteck

In [90]:
import tempfile
import os

def get_cantera_mechanism_from_model(kinetic_model):
    """
    Export a KineticModel from the database to a Cantera YAML file.
    Returns the path to the temporary YAML file.
    """
    try:
        result = build_cantera_yaml(kinetic_model)
        
        # Save to a temporary file
        temp_dir = tempfile.mkdtemp()
        yaml_path = os.path.join(temp_dir, result.filename)
        
        with open(yaml_path, 'wb') as f:
            f.write(result.content)
        
        return yaml_path
        
    except Exception as e:
        print(f"Error exporting model: {e}")
        return None


In [91]:
# More robust search: by SMILES, InChI, or InChIKey
from django.db.models import Q

def get_datasets_by_fuel_identifier(smiles=None, inchi=None, inchikey=None, experiment_type=None):
    """
    Find all ExperimentDatasets that contain a species with matching identifiers.
    """
    # Build query for CompositionSpecies
    query = Q()
    if smiles:
        query |= Q(smiles=smiles)
    if inchi:
        query |= Q(inchi=inchi)
    if inchikey:
        query |= Q(inchi__icontains=inchikey)
    
    matching_species = CompositionSpecies.objects.filter(query)
    
    # Get unique dataset IDs
    dataset_ids = set()
    for sp in matching_species:
        if sp.composition:
            for dp in sp.composition.datapoints.all():
                dataset_ids.add(dp.dataset_id)
    
    datasets = ExperimentDataset.objects.filter(id__in=dataset_ids).distinct()
    if experiment_type:
        datasets = datasets.filter(experiment_type=experiment_type)
    
    return datasets

In [92]:
# Find kinetic models that contain species matching the experiment identifiers
from database.models import KineticModel, Species, Isomer, Structure

def get_kinetic_models_by_species_identifier(smiles=None, inchi=None, inchikey=None):
    """
    Find all KineticModels that contain a species matching the given identifiers.
    """
    # Find matching isomers by InChI
    isomer_ids = set()
    if inchi:
        isomer_ids.update(Isomer.objects.filter(inchi__icontains=inchi).values_list('id', flat=True))
    if inchikey:
        isomer_ids.update(Isomer.objects.filter(inchi__icontains=inchikey).values_list('id', flat=True))
    
    # Find matching structures by SMILES
    if smiles:
        # Also get isomer_ids from matching structures
        for struct in Structure.objects.filter(smiles=smiles):
            isomer_ids.add(struct.isomer_id)
    
    # Find species that have these isomers
    species_ids = Species.objects.filter(isomers__id__in=isomer_ids).values_list('id', flat=True)
    
    # Find kinetic models that contain these species
    kinetic_models = KineticModel.objects.filter(species__id__in=species_ids).distinct()
    
    return kinetic_models


In [93]:
# n-heptane identifiers
N_HEPTANE_SMILES = 'CCCCCCC'
N_HEPTANE_INCHI = '1S/C7H16/c1-3-5-7-6-4-2/h3-7H2,1-2H3'
N_HEPTANE_INCHIKEY = 'IMNFDUFMRHMDMM'

In [94]:
# Search using all identifiers
all_n_heptane_dataset = get_datasets_by_fuel_identifier(
    smiles=N_HEPTANE_SMILES,
    inchi=N_HEPTANE_INCHI,
    inchikey=N_HEPTANE_INCHIKEY,
    experiment_type='ignition delay'
)
print(all_n_heptane_dataset)

for dataset in all_n_heptane_dataset:
    first_dp = dataset.datapoints.first()
    target = first_dp.get_ignition_target() if first_dp else None
    print(f"{dataset.chemked_file_path}, target: {target}")

<QuerySet [<ExperimentDataset: n-heptane/Vermeer 1972/st_vermeer_1972.yaml (72 points)>, <ExperimentDataset: n-heptane/Burcat 1981/st_burcat_1981-1.yaml (18 points)>, <ExperimentDataset: n-heptane/Burcat 1981/st_burcat_1981-2.yaml (6 points)>, <ExperimentDataset: n-heptane/Burcat 1981/st_burcat_1981-3.yaml (6 points)>, <ExperimentDataset: n-heptane/Burcat 1981/st_burcat_1981-4.yaml (6 points)>, <ExperimentDataset: n-heptane/Burcat 1981/st_burcat_1981-5.yaml (6 points)>, <ExperimentDataset: n-heptane/Ciezki 1993/st_ciezki_1993-1.yaml (14 points)>, <ExperimentDataset: n-heptane/Ciezki 1993/st_ciezki_1993-2.yaml (12 points)>, <ExperimentDataset: n-heptane/Ciezki 1993/st_ciezki_1993-3.yaml (16 points)>, <ExperimentDataset: n-heptane/Ciezki 1993/st_ciezki_1993-4.yaml (52 points)>, <ExperimentDataset: n-heptane/Ciezki 1993/st_ciezki_1993-5.yaml (12 points)>, <ExperimentDataset: n-heptane/Ciezki 1993/st_ciezki_1993-6.yaml (8 points)>, <ExperimentDataset: n-heptane/Ciezki 1993/st_ciezki_1993-7

In [95]:
# Search for kinetic models containing n-heptane
n_heptane_models = list(get_kinetic_models_by_species_identifier(
    smiles=N_HEPTANE_SMILES,
    inchi=N_HEPTANE_INCHI,
    inchikey=N_HEPTANE_INCHIKEY
))

print(f"Found {len(n_heptane_models)} kinetic models containing n-heptane:")
for model in n_heptane_models:
    print(f"{model.id}  - {model.model_name}")

Found 6 kinetic models containing n-heptane:
6  - n-Heptane
23  - Narayanaswamy
30  - Gasoline_2
40  - Gasoline_Surrogate
43  - Biomass
44  - EL24115


In [96]:
# get full file path for a ChemKED dataset
from django.conf import settings

def _find_file_in_tree(root_dir, rel_path):
    """Try to locate rel_path under root_dir."""
    candidate = os.path.join(root_dir, rel_path)
    if os.path.exists(candidate):
        return candidate

    # Fallback: search by filename in tree
    filename = os.path.basename(rel_path)
    for dirpath, _, filenames in os.walk(root_dir):
        if filename in filenames:
            return os.path.join(dirpath, filename)

    return None


def get_chemked_dataset_path(dataset, base_dir=None, search_dirs=None):
    """
    Return the full file path for a ChemKED dataset.

    Args:
        dataset: ExperimentDataset instance or a chemked_file_path string
        base_dir: Optional base directory override
        search_dirs: Optional list of directories to search
    """
    rel_path = dataset.chemked_file_path if hasattr(dataset, "chemked_file_path") else str(dataset)

    if base_dir is None:
        # Prefer explicit env/setting if available
        base_dir = (
            getattr(settings, "CHEMKED_DATA_ROOT", None)
            or os.getenv("CHEMKED_DATA_ROOT")
        )

    # Fallbacks if no explicit base_dir provided
    if not base_dir:
        candidates = [
            os.path.join(PROJECT_ROOT, "chemked_database", "chemked_data"),
            os.path.join(PROJECT_ROOT, "chemked_database", "results"),
            os.path.join(PROJECT_ROOT, "..", "ChemKED-database"),
            os.path.join(getattr(settings, "MEDIA_ROOT", ""), "chemked"),
            os.path.join(getattr(settings, "MEDIA_ROOT", ""), "chemked_database"),
        ]
    else:
        candidates = [base_dir]

    # Add any explicit search dirs
    if search_dirs:
        candidates.extend(search_dirs)

    # Resolve first match
    for root_dir in [c for c in candidates if c]:
        root_dir = os.path.abspath(root_dir)
        if os.path.exists(root_dir):
            match = _find_file_in_tree(root_dir, rel_path)
            if match:
                return match

    return None

# Example usage:
# dataset_path = get_chemked_dataset_path(selected_dataset)
# print(dataset_path or "Dataset file not found")

In [97]:
selected_dataset = all_n_heptane_dataset.filter(chemked_file_path='n-heptane/Vermeer 1972/st_vermeer_1972.yaml')
print(selected_dataset)

<QuerySet [<ExperimentDataset: n-heptane/Vermeer 1972/st_vermeer_1972.yaml (72 points)>]>


In [98]:
selected_dataset = all_n_heptane_dataset.filter(
    chemked_file_path='n-heptane/Vermeer 1972/st_vermeer_1972.yaml'
).first()

print(selected_dataset.chemked_file_path)

dataset_path = get_chemked_dataset_path(selected_dataset)
print(dataset_path)

n-heptane/Vermeer 1972/st_vermeer_1972.yaml
/Users/lekiaprosper/Documents/CoMoChEng/Prometheus/ChemKED-database/n-heptane/Vermeer 1972/st_vermeer_1972.yaml


In [99]:
from pyteck.eval_model import evaluate_model

In [100]:
selected_model = KineticModel.objects.get(id=6)
mechanism_path = get_cantera_mechanism_from_model(selected_model)

In [101]:
print(mechanism_path)

/var/folders/s3/w8v7qhm16dn7t5xvxlk9n1bc0000gn/T/tmp1dy38jyt/n-heptane.yaml


In [102]:
from collections import OrderedDict

def _best_species_label(sn):
    """Pick a stable label: prefer SpeciesName, then species.names, then formula, then id."""
    if sn.name:
        return sn.name
    # Try any associated names on the species
    names = sorted(getattr(sn.species, "names", []) or [])
    if names:
        return names[0]
    # Try chemical formula
    formula = getattr(sn.species, "formula", None)
    if formula:
        return formula
    return str(sn.species_id)


def _normalize_label(label):
    return "".join(ch for ch in label.upper() if ch.isalnum())


def _best_normalized_match(target, normalized_candidates):
    """Find candidate whose normalized name contains target (or vice versa)."""
    if not target:
        return None
    for norm, original in normalized_candidates.items():
        if target in norm or norm in target:
            return original
    return None


def get_species_label_smiles_map(model):
    """
    Return OrderedDict of {label: smiles} for a KineticModel.
    Uses SpeciesName (model-specific label) and Structure.smiles if available.
    """
    label_smiles = OrderedDict()

    # SpeciesName provides the model-specific label
    for sn in SpeciesName.objects.filter(kinetic_model=model).select_related("species"):
        label = _best_species_label(sn)
        smiles = None

        # Prefer first non-empty SMILES from linked structures
        for struct in sn.species.structures:
            if getattr(struct, "smiles", None):
                smiles = struct.smiles
                break

        # Fallback to label if SMILES missing
        label_smiles[label] = smiles or label

    return label_smiles


def get_dataset_species_map(dataset):
    """Return {species_name: smiles_or_name} for a ChemKED dataset."""
    species_map = OrderedDict()
    if not dataset:
        return species_map
    # Use composition from first datapoint
    dp = dataset.datapoints.first()
    if not dp or not dp.composition:
        return species_map

    for sp in dp.composition.species.all():
        key = sp.species_name
        value = sp.smiles or sp.species_name
        species_map[key] = value

    return species_map


def build_spec_keys_for_dataset(model, dataset, model_species_names=None):
    """
    Build mapping of dataset species -> model species label.
    Uses SMILES matching when available, then normalized label match.
    Ensures output species labels exist in the mechanism when provided.
    """
    model_label_smiles = get_species_label_smiles_map(model)
    smiles_to_label = {}
    for label, smiles in model_label_smiles.items():
        if smiles:
            smiles_to_label[smiles] = label

    normalized_model_names = {}
    if model_species_names:
        for name in model_species_names:
            normalized_model_names[_normalize_label(name)] = name

    dataset_species = get_dataset_species_map(dataset)
    spec_keys = OrderedDict()
    for ds_label, ds_smiles in dataset_species.items():
        ds_norm = _normalize_label(ds_label)

        # Prefer SMILES match
        model_label = smiles_to_label.get(ds_smiles)

        # If SMILES match gives a label not in mechanism, try normalized match to actual
        if model_label and model_species_names and model_label not in model_species_names:
            model_label = (
                normalized_model_names.get(_normalize_label(model_label))
                or _best_normalized_match(_normalize_label(model_label), normalized_model_names)
                or model_label
            )

        # Fallback: normalized label match against mechanism species names
        if not model_label and model_species_names:
            model_label = normalized_model_names.get(ds_norm)

        # Fallback: substring normalized match (e.g., nC7H16 -> C7H16)
        if not model_label and model_species_names:
            model_label = _best_normalized_match(ds_norm, normalized_model_names)

        # Fallback: if dataset label matches a model label
        if not model_label and ds_label in model_label_smiles:
            model_label = ds_label

        # Final fallback: keep dataset label (may error later)
        spec_keys[ds_label] = model_label or ds_label

    return spec_keys


def write_spec_keys_yaml(models, output_path="spec_keys.yaml", extra_species=None, model_name_map=None, dataset=None, model_species_names=None):
    """Write spec_keys.yaml in the requested format."""
    extra_species = extra_species or {}
    model_name_map = model_name_map or {}
    with open(output_path, "w", encoding="utf-8") as f:
        for model in models:
            model_key = model_name_map.get(model.model_name, model.model_name)
            f.write(f"{model_key}:\n")

            if dataset is not None:
                label_smiles = build_spec_keys_for_dataset(model, dataset, model_species_names=model_species_names)
            else:
                label_smiles = get_species_label_smiles_map(model)
                # Merge in any extra species (dataset labels) if missing
                for label, smiles in extra_species.items():
                    label_smiles.setdefault(label, smiles)

            for label, smiles in label_smiles.items():
                f.write(f"    {label}: \"{smiles}\"\n")

    return output_path

In [103]:
import cantera as ct

models_to_report = KineticModel.objects.filter(model_name__in=["n-Heptane", "Narayanaswamy", "Gasoline_2", "Gasoline_Surrogate", "Biomass", "EL24115"])

# Map model name to the actual mechanism filename used by PyTeCK
model_name_map = {
    "n-Heptane": "n-heptane.yaml",
}

# Load mechanism to get actual species names for mapping
model_for_mapping = models_to_report.filter(model_name="n-Heptane").first()
mechanism_path = get_cantera_mechanism_from_model(model_for_mapping)
solution = ct.Solution(mechanism_path)
model_species_names = list(solution.species_names)

# Build spec_keys.yaml using dataset species -> model labels
write_spec_keys_yaml(
    models_to_report,
    output_path="spec_keys.yaml",
    model_name_map=model_name_map,
    dataset=selected_dataset,
    model_species_names=model_species_names,
)
print("Wrote spec_keys.yaml")

Wrote spec_keys.yaml


In [104]:
# Debug: show mechanism species containing C7H16 and the spec_keys entry
print("Mechanism species containing C7H16:")
for name in model_species_names:
    if "C7H16" in name.upper():
        print("  ", name)

# Show the mapping for dataset species
spec_keys_debug = build_spec_keys_for_dataset(model_for_mapping, selected_dataset, model_species_names=model_species_names)
print("nC7H16 mapping:", spec_keys_debug.get("nC7H16"))

Mechanism species containing C7H16:
   N-C7H1601(1017)
   C7H16O2(1023)
   C7H16O2(1025)
   C7H16O2(1027)
   C7H16O2(1029)
nC7H16 mapping: N-C7H1601(1017)
nC7H16 mapping: N-C7H1601(1017)


In [105]:
# Build a dataset list file for PyTeCK (expects relative paths under data_path)
selected_dataset = all_n_heptane_dataset.filter(
    chemked_file_path='n-heptane/Vermeer 1972/st_vermeer_1972.yaml'
).first()

# ChemKED root directory
chemked_root = os.path.abspath(os.path.join(PROJECT_ROOT, "..", "ChemKED-database"))

# Write a dataset list file (one relative path per line)
dataset_list_file = "dataset_file.txt"
with open(dataset_list_file, "w", encoding="utf-8") as f:
    f.write(f"{selected_dataset.chemked_file_path}\n")

# Use the mechanism file generated from the selected model
model_file = os.path.basename(mechanism_path)
model_dir = os.path.dirname(mechanism_path)

# Evaluate model (dataset_file contains relative paths; data_path is the root)
evaluate_model(
    model_name=model_file,
    spec_keys_file='spec_keys.yaml',
    dataset_file=dataset_list_file,
    data_path=chemked_root,
    model_path=model_dir,
    results_path='results',
    skip_validation=True,
)

Done with case  st_vermeer_1972_6
Done with case  st_vermeer_1972_6
Done with case  st_vermeer_1972_2
Done with case  st_vermeer_1972_2
Done with case  st_vermeer_1972_3
Done with case  st_vermeer_1972_5
Done with case  st_vermeer_1972_3
Done with case  st_vermeer_1972_5
Done with case  st_vermeer_1972_1
Done with case  st_vermeer_1972_1
Done with case  st_vermeer_1972_4
Done with case  st_vermeer_1972_0
Done with case  st_vermeer_1972_4
Done with case  st_vermeer_1972_0
Done with case  st_vermeer_1972_7
Done with case  st_vermeer_1972_7
Done with case  st_vermeer_1972_8
Done with case  st_vermeer_1972_8
Done with case  st_vermeer_1972_13
Done with case  st_vermeer_1972_13
Done with case  st_vermeer_1972_15
Done with case  st_vermeer_1972_15
Done with case  st_vermeer_1972_9
Done with case  st_vermeer_1972_9
Done with case  st_vermeer_1972_14
Done with case  st_vermeer_1972_14
Done with case  st_vermeer_1972_16
Done with case  st_vermeer_1972_16
Done with case  st_vermeer_1972_12
Done 

{'model': 'n-heptane.yaml',
 'datasets': [{'dataset': 'n-heptane/Vermeer 1972/st_vermeer_1972.yaml',
   'dataset_id': 0,
   'standard deviation': 0.21658746091694067,
   'datapoints': [{'experimental ignition delay': '1.3e-05 second',
     'simulated ignition delay': '1.3860563345905778e-06 second',
     'temperature': '1580.0 kelvin',
     'pressure': '269524.5 pascal',
     'composition': [{'InChI': '1S/C7H16/c1-3-5-7-6-4-2/h3-7H2,1-2H3',
       'species-name': 'nC7H16',
       'amount': '0.025'},
      {'InChI': '1S/O2/c1-2', 'species-name': 'O2', 'amount': '0.275'},
      {'InChI': '1S/Ar', 'species-name': 'Ar', 'amount': '0.7'}],
     'composition type': 'mole fraction'},
    {'experimental ignition delay': '1.8e-05 second',
     'simulated ignition delay': '1.650446821465796e-06 second',
     'temperature': '1555.0 kelvin',
     'pressure': '259392.0 pascal',
     'composition': [{'InChI': '1S/C7H16/c1-3-5-7-6-4-2/h3-7H2,1-2H3',
       'species-name': 'nC7H16',
       'amount': '

In [106]:


# Example usage:
# models_to_report = KineticModel.objects.filter(model_name__in=["gri30", "h2o2"])
# write_spec_keys_yaml(models_to_report, output_path="spec_keys.yaml")
# print("Wrote spec_keys.yaml")